
# Classical baseline models

Training a few classical models on the churn dataset using the existing `feature_pipeline.py` utilities to keep preprocessing consistent. The notebook builds user-level features from event-level records, then fits linear/logistic regression, decision tree, and random forest baselines. Each model also produces a submission CSV.


In [1]:

import os, sys
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from submission_utils import fit_and_save_submission, fit_and_save_many
sys.path.append(os.getcwd())
from feature_pipeline import prepare_datasets


In [2]:

BASE_PATH = os.getcwd()
train_path = f"{BASE_PATH}/churn-prediction-25-26/train.parquet"
test_path = f"{BASE_PATH}/churn-prediction-25-26/test.parquet"

train_df, val_df, test_df, labels, artifacts = prepare_datasets(
    train_path,
    test_path,
    val_ratio=0.2,
    random_state=42,
)
print(f"train rows: {len(train_df):,}, val rows: {len(val_df):,}, test rows: {len(test_df):,}")
print(f"unique train users: {train_df.userId.nunique()}, val users: {val_df.userId.nunique()}")


100%|██████████| 4393179/4393179 [00:18<00:00, 232613.55it/s]


train rows: 12,630,959, val rows: 3,122,857, test rows: 4,393,179
unique train users: 15312, val users: 3828


In [3]:

def aggregate_user_features(df: pd.DataFrame, numeric_cols):
    # Aggregate event-level rows to user-level summary stats.
    grouped = df.groupby("userId")
    num_stats = grouped[numeric_cols].agg(["mean", "std", "min", "max"])
    num_stats.columns = [f"{c}_{stat}" for c, stat in num_stats.columns]

    def most_frequent(series: pd.Series) -> int:
        return int(series.value_counts().index[0]) if len(series) else 0

    cat_mode = grouped[["page_id", "metro_id", "state_id", "device_id"]].agg(most_frequent)
    lengths = grouped.size().rename("event_count")

    features = pd.concat([num_stats, cat_mode, lengths], axis=1)
    return features.fillna(0)

X_train = aggregate_user_features(train_df, artifacts.numeric_cols)
X_val = aggregate_user_features(val_df, artifacts.numeric_cols)

y_train = labels.loc[X_train.index]
y_val = labels.loc[X_val.index]

print(f"Train feature shape: {X_train.shape}, Val feature shape: {X_val.shape}")


Train feature shape: (15312, 65), Val feature shape: (3828, 65)


In [4]:

models: Dict[str, object] = {
    "linear_regression": Pipeline(
        [("scaler", StandardScaler()), ("model", LinearRegression())]
    ),
    "logistic_regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    max_iter=1000, class_weight="balanced", solver="lbfgs"
                ),
            ),
        ]
    ),
    "decision_tree": DecisionTreeClassifier(
        max_depth=12, min_samples_leaf=5, class_weight="balanced", random_state=42
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=3,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42,
    ),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    if hasattr(model, "predict_proba"):
        val_proba = model.predict_proba(X_val)[:, 1]
    else:
        val_proba = np.clip(model.predict(X_val), 0, 1)
    val_pred = (val_proba >= 0.5).astype(int)

    results.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_val, val_pred),
            "roc_auc": roc_auc_score(y_val, val_proba),
            "f1": f1_score(y_val, val_pred),
        }
    )

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
results_df


,model,accuracy,roc_auc,f1
1,logistic_regression,0.763584,0.816347,0.583525
3,random_forest,0.821317,0.812908,0.573034
0,linear_regression,0.811912,0.812288,0.447853
2,decision_tree,0.733020,0.706571,0.508654


In [5]:
# Refit each model on full train+val and write a submission CSV for each.
sub_dir = Path(BASE_PATH) / "Submission"
sub_dir.mkdir(parents=True, exist_ok=True)

# To convert probabilities to class labels
threshold = 0.5 

X_full = aggregate_user_features(pd.concat([train_df, val_df], axis=0), artifacts.numeric_cols)
y_full = labels.loc[X_full.index]
X_test = aggregate_user_features(test_df, artifacts.numeric_cols)

saved_paths = fit_and_save_many(models, X_full, y_full, X_test, sub_dir, threshold=threshold)
saved_paths


Saved /Users/martino.orioligmail.com/datascience/Submission/submission_linear_regression.csv
Saved /Users/martino.orioligmail.com/datascience/Submission/submission_logistic_regression.csv
Saved /Users/martino.orioligmail.com/datascience/Submission/submission_decision_tree.csv
Saved /Users/martino.orioligmail.com/datascience/Submission/submission_random_forest.csv


[PosixPath('/Users/martino.orioligmail.com/datascience/Submission/submission_linear_regression.csv'),
 PosixPath('/Users/martino.orioligmail.com/datascience/Submission/submission_logistic_regression.csv'),
 PosixPath('/Users/martino.orioligmail.com/datascience/Submission/submission_decision_tree.csv'),
 PosixPath('/Users/martino.orioligmail.com/datascience/Submission/submission_random_forest.csv')]

## Logistic Regression (GridSearchCV)

In [6]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

logit_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, solver="liblinear")),
    ]
)

param_grid = {
    "model__C": [0.01, 0.1, 1.0, 10.0],
    "model__penalty": ["l1", "l2"],
    "model__class_weight": [None, "balanced"],
}

# Stratified CV keeps class balance while shuffling similarly to the existing split.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=logit_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
)

grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print(f"Mean CV ROC-AUC: {grid_search.best_score_:.4f}")

best_logit = grid_search.best_estimator_
val_proba = best_logit.predict_proba(X_val)[:, 1]
print(f"Validation ROC-AUC: {roc_auc_score(y_val, val_proba):.4f}")

val_pred_logit = pd.Series(val_proba, index=X_val.index, name="logit_grid_val_proba")
test_features = X_test if "X_test" in globals() else aggregate_user_features(test_df, artifacts.numeric_cols)
test_pred_logit = pd.Series(
    best_logit.predict_proba(test_features)[:, 1],
    index=test_features.index,
    name="logit_grid_test_proba",
)

{
    "val_pred_sample": val_pred_logit.head(),
    "test_pred_sample": test_pred_logit.head(),
}


Best params: {'model__C': 1.0, 'model__class_weight': 'balanced', 'model__penalty': 'l1'}
Mean CV ROC-AUC: 0.8322
Validation ROC-AUC: 0.8167


{'val_pred_sample': userId
 1000103    0.360506
 1000182    0.423844
 1000194    0.669052
 1000248    0.281930
 1000409    0.291886
 Name: logit_grid_val_proba, dtype: float64,
 'test_pred_sample': userId
 1000655    0.077867
 1000963    0.288622
 1001129    0.080681
 1001963    0.208735
 1002283    0.319194
 Name: logit_grid_test_proba, dtype: float64}

In [ ]:
logit_grid_path = fit_and_save_submission(
    best_logit,
    X_full,
    y_full,
    X_test,
    sub_dir / "submission_logit_grid.csv",
    threshold=threshold,
)

logit_grid_path
